# XAI - Decodability in feature space (Analysis 3): do the attributed patches carry signature-specific morphology?

Analyses 1 (spatial specificity) and 2 (mass) were **descriptive on the attribution map** and came
out almost empty. Here we change the question to a **supervised / faithfulness** test:

> Do the **features** of the top-K patches the model attributes to signature X **predict the genomic
> presence of X**? And do they do so **better** than generic patches (slide mean / random) and than
> the top-K of a **different** signature?

For each patient we build feature profiles (mean of the UNI-1536 patch vectors):
- `topX[s]` = mean of the top-K patches attributed to signature s (one per each of the 30 signatures);
- `slide-mean` = mean over all patches; `random` = mean of K random patches.

Then, for each target X, we train a **linear probe** (StandardScaler -> PCA -> LogisticRegression) in
**cross-validation** to predict plus/minus for X, and measure the **ROC-AUC**. Two comparisons:
- **AUC(topX) vs AUC(random/slide-mean)** -> the attribution concentrates *informative* patches;
- **rank of topX among the 30 signatures** -> if the attribution is *X-specific*, X's patches should
  be the best predictor of X (rank close to 1).

If topX predicts X above chance and better than the other signatures, there **is signal** that the
descriptive analyses were missing. Everything is **GPU-free** (it reads the local `.pt` files).


## 0. Config

In [ ]:
import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold

GRAD_DIR     = '../grad'
OUT_DIR      = os.path.join(GRAD_DIR, 'xai_outputs')
SPLIT_DIR    = '../coad_stad_explainability_4fold'
FEATURES_DIR = '../features'
os.makedirs(OUT_DIR, exist_ok=True)

TISSUES = {'coad': ['SBS18', 'SBS44', 'SBS93'],
           'stad': ['SBS8', 'SBS18', 'SBS40a', 'SBS44']}
K        = 100      # top-K patch per firma
N_PCA    = 50       # riduzione prima del probe (stabilita' con d=1536, n~200)
N_SPLITS = 5
SEED     = 42
np.random.seed(SEED)
LABEL    = 'plus_minus'   # 'plus_minus' (genomica) oppure 'TP_TN'
print('K=', K, '| PCA=', N_PCA, '| label=', LABEL)

## 1. Helpers: per-patient feature profiles (precomputed from the .pt files if missing), and the probe


In [ ]:
def profiles_path(tissue):
    return os.path.join(OUT_DIR, f'feat_profiles_{tissue}_K{K}.npz')

def build_profiles(tissue):
    """Precompute (GPU-free): per patient, mean feature of the top-K patches of EVERY signature,
    plus slide-mean and random-K. Reads the pooled cache (feature_file+original_index) and the .pt files."""
    import pyarrow.parquet as pq, torch
    p = os.path.join(OUT_DIR, f'attr_{tissue}_subset_scores.parquet')
    cols = ['sample_id', 'case_id', 'feature_file', 'signature_name', 'original_index', 'grad_score']
    tbl = pq.read_table(p, columns=cols)
    for c in ['sample_id', 'case_id', 'feature_file', 'signature_name']:
        i = tbl.schema.get_field_index(c); tbl = tbl.set_column(i, c, tbl.column(c).dictionary_encode())
    a = tbl.to_pandas(); a['ag'] = np.abs(a['grad_score'].astype('float32'))
    sigs = sorted(a['signature_name'].astype(str).unique()); sidx = {s: i for i, s in enumerate(sigs)}
    cases = sorted(a['case_id'].astype(str).unique()); cidx = {c: i for i, c in enumerate(cases)}
    nC, nS, D = len(cases), len(sigs), 1536
    topsum = np.zeros((nC, nS, D)); topcnt = np.zeros((nC, nS), 'int32')
    msum = np.zeros((nC, D)); rsum = np.zeros((nC, D)); scnt = np.zeros(nC, 'int32')
    rng = np.random.default_rng(SEED)
    for sid, g in a.groupby('sample_id', observed=True):
        ff = str(g['feature_file'].iloc[0]); ci = cidx[str(g['case_id'].iloc[0])]
        try:
            feat = torch.load(os.path.join(FEATURES_DIR, ff), map_location='cpu', weights_only=True)['feat'].float().numpy()
        except Exception:
            feat = torch.load(os.path.join(FEATURES_DIR, ff), map_location='cpu')['feat'].float().numpy()
        L = feat.shape[0]; msum[ci] += feat.mean(0); scnt[ci] += 1
        rsum[ci] += feat[rng.choice(L, min(K, L), replace=False)].mean(0)
        for sig, gs in g.groupby('signature_name', observed=True):
            oi = gs.nlargest(K, 'ag')['original_index'].to_numpy(); oi = oi[oi < L]
            if len(oi):
                topsum[ci, sidx[str(sig)]] += feat[oi].mean(0); topcnt[ci, sidx[str(sig)]] += 1
    topX = (topsum / np.maximum(topcnt, 1)[:, :, None]).astype('float32')
    meanf = (msum / np.maximum(scnt, 1)[:, None]).astype('float32')
    randf = (rsum / np.maximum(scnt, 1)[:, None]).astype('float32')
    np.savez_compressed(profiles_path(tissue), topX=topX, meanf=meanf, randf=randf,
                        cases=np.array(cases), sigs=np.array(sigs), topcnt=topcnt)
    print(f'[{tissue}] profili costruiti: {nC} casi x {nS} firme')

def load_profiles(tissue):
    if not os.path.exists(profiles_path(tissue)):
        build_profiles(tissue)
    d = np.load(profiles_path(tissue), allow_pickle=True)
    sigs = [str(s) for s in d['sigs']]; cases = [str(c) for c in d['cases']]
    return {'topX': d['topX'], 'meanf': d['meanf'], 'randf': d['randf'],
            'sigs': sigs, 'cases': cases, 'sidx': {s: i for i, s in enumerate(sigs)}}

def labels_for(tissue, target, cases):
    sp = pd.read_csv(os.path.join(SPLIT_DIR, f'{tissue}_{target}.csv'))
    if LABEL == 'TP_TN' and 'status' in sp.columns:
        m = dict(zip(sp['Case ID'], np.where(sp['status'] == 'TP', 1, np.where(sp['status'] == 'TN', 0, -1))))
    else:
        m = dict(zip(sp['Case ID'], np.where(sp['group'] == f'{target}_plus', 1, 0)))
    return np.array([m.get(c, -1) for c in cases])

def probe_auc(Xmat, y, seed=SEED):
    m = y >= 0; Xx = Xmat[m]; yy = y[m]
    if len(np.unique(yy)) < 2 or min(np.bincount(yy)) < N_SPLITS:
        return np.nan
    npca = int(min(N_PCA, Xx.shape[0] - N_SPLITS, Xx.shape[1]))
    pipe = make_pipeline(StandardScaler(), PCA(n_components=npca, random_state=seed),
                         LogisticRegression(max_iter=1000, class_weight='balanced'))
    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    return float(cross_val_score(pipe, Xx, yy, cv=cv, scoring='roc_auc').mean())

## 2. Probe: do X's top-K patches predict X? (and better than other signatures / chance?)
For each target: the AUC of the target signature, its **rank** among all 30, and the slide-mean /
random baselines. Green in the plot marks the target.


In [ ]:
rows = []
for tissue, targets in TISSUES.items():
    P = load_profiles(tissue); sigs = P['sigs']; cases = P['cases']
    for target in targets:
        y = labels_for(tissue, target, cases)
        aucs = {s: probe_auc(P['topX'][:, P['sidx'][s], :], y) for s in sigs}
        auc_mean = probe_auc(P['meanf'], y); auc_rand = probe_auc(P['randf'], y)
        ser = pd.Series(aucs).sort_values(ascending=False)
        auc_tx = aucs[target]; rank = int(np.where(ser.index == target)[0][0]) + 1
        rows.append({'tissue': tissue, 'target': target,
                     'n_pos': int((y == 1).sum()), 'n_neg': int((y == 0).sum()),
                     'AUC_topX': round(auc_tx, 3), 'rank_topX_/30': rank,
                     'AUC_best': round(ser.iloc[0], 3), 'best_sig': ser.index[0],
                     'AUC_slide_mean': round(auc_mean, 3), 'AUC_random': round(auc_rand, 3),
                     'gain_vs_mean': round(auc_tx - auc_mean, 3)})
        fig, ax = plt.subplots(figsize=(9, 4.5))
        col = ['#2ca02c' if s == target else '#bbbbbb' for s in ser.index]
        ax.bar(range(len(ser)), ser.values, color=col)
        ax.axhline(0.5, color='k', ls=':', lw=1)
        ax.axhline(auc_mean, color='#1f77b4', ls='--', lw=1, label=f'slide-mean {auc_mean:.2f}')
        ax.axhline(auc_rand, color='#ff7f0e', ls='--', lw=1, label=f'random {auc_rand:.2f}')
        ax.set_xticks(range(len(ser))); ax.set_xticklabels(ser.index, rotation=90, fontsize=6)
        ax.set_ylabel(f'CV ROC-AUC (predice {target})')
        ax.set_title(f'{tissue.upper()} {target}: AUC per firma delle top-{K} patch '
                     f'(verde=target, rank {rank}/30, n+={int((y==1).sum())})')
        ax.legend(fontsize=8); plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR, f'{tissue}_{target}_decodability.png'), dpi=150, bbox_inches='tight')
        plt.show()
res = pd.DataFrame(rows)
res.to_csv(os.path.join(OUT_DIR, 'decodability_ALL.csv'), index=False)
res

## 3. Reading the result
- **AUC(topX) ~ 0.5** -> the patches attributed to X do not separate present from absent: no signal.
- **AUC(topX) > 0.5 and > slide-mean/random** -> the attribution concentrates patches that are
  *informative* for X.
- **rank of topX close to 1** (out of 30) -> the signal is *X-specific*: X's patches predict X better
  than the patches attributed to the other signatures. If the rank is high instead (X is not among
  the best), the predictive power is not specific to X's attribution.
- Compare `gain_vs_mean`: how much the top-K patches beat the plain slide mean. If it is around 0,
  the attribution adds nothing over looking at the whole slide.

Note: the probe is linear on PCA-50 (stable with d=1536 and small n); the AUC is from 5-fold CV.
Small n (SBS93/SBS8) makes the estimates noisy. Set `LABEL='TP_TN'` for a cleaner contrast, at the
cost of fewer cases.
